In [1]:
import pandas as pd

# Load Data

In [3]:
customer = pd.read_csv('customers.csv')

In [4]:
order = pd.read_csv('orders.csv')

In [5]:
product = pd.read_csv('products.csv')

In [6]:
customer

,CustomerID,CustomerName,City
0,1,John,Colombo
1,2,Sarah,Kandy
2,3,Mike,Galle
3,4,Emma,Jaffna
4,5,David,Colombo


In [7]:
order

,OrderID,CustomerID,ProductID,Quantity
0,1001,1,101,1
1,1002,2,102,2
2,1003,1,103,1
3,1004,3,104,1
4,1005,4,105,2
5,1006,5,101,1
6,1007,2,105,1
7,1008,3,102,3


In [8]:
product

,ProductID,ProductName,Price
0,101,Laptop,250000
1,102,Mouse,3000
2,103,Keyboard,8000
3,104,Monitor,50000
4,105,Headphones,12000


# Merge Orders and Customers

In [9]:
order_customer = pd.merge(order,customer,left_on='CustomerID',right_on='CustomerID',how='left')

Although when column names are the same, you can simplify:
```bash
pd.merge(order, customer, on='CustomerID', how='left')
```

In [10]:
order_customer

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City
0,1001,1,101,1,John,Colombo
1,1002,2,102,2,Sarah,Kandy
2,1003,1,103,1,John,Colombo
3,1004,3,104,1,Mike,Galle
4,1005,4,105,2,Emma,Jaffna
5,1006,5,101,1,David,Colombo
6,1007,2,105,1,Sarah,Kandy
7,1008,3,102,3,Mike,Galle


# Merge Products

In [12]:
sales = pd.merge(order_customer,product,left_on='ProductID',right_on='ProductID',how='left')

In [13]:
sales

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price
0,1001,1,101,1,John,Colombo,Laptop,250000
1,1002,2,102,2,Sarah,Kandy,Mouse,3000
2,1003,1,103,1,John,Colombo,Keyboard,8000
3,1004,3,104,1,Mike,Galle,Monitor,50000
4,1005,4,105,2,Emma,Jaffna,Headphones,12000
5,1006,5,101,1,David,Colombo,Laptop,250000
6,1007,2,105,1,Sarah,Kandy,Headphones,12000
7,1008,3,102,3,Mike,Galle,Mouse,3000


# Create Total Amount

In [14]:
sales["TotalAmount"] = sales['Quantity'] * sales['Price']

In [15]:
sales

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price,TotalAmount
0,1001,1,101,1,John,Colombo,Laptop,250000,250000
1,1002,2,102,2,Sarah,Kandy,Mouse,3000,6000
2,1003,1,103,1,John,Colombo,Keyboard,8000,8000
3,1004,3,104,1,Mike,Galle,Monitor,50000,50000
4,1005,4,105,2,Emma,Jaffna,Headphones,12000,24000
5,1006,5,101,1,David,Colombo,Laptop,250000,250000
6,1007,2,105,1,Sarah,Kandy,Headphones,12000,12000
7,1008,3,102,3,Mike,Galle,Mouse,3000,9000


# Total Revenue

In [16]:
sales['TotalAmount'].sum()

np.int64(609000)

# Customer Spending

In [17]:
sales.groupby('CustomerID').agg({'TotalAmount':'sum'})

,TotalAmount
CustomerID,
1,258000
2,18000
3,59000
4,24000
5,250000


# Product Performance

In [20]:
sales.groupby('ProductID').agg({'Quantity':'sum'}).sort_values(by='Quantity',ascending=False)

,Quantity
ProductID,
102,5
105,3
101,2
103,1
104,1


In [30]:
sales.groupby('ProductID').agg({'Quantity':'sum'}).idxmax()

Quantity    102
dtype: int64

In [31]:
sales.groupby('ProductID').agg({'Quantity':'sum'}).idxmin()

Quantity    103
dtype: int64

- Since the dataset is small i can see the max and min values 
- But if it is large how can i detect 103 and 104 both are min ?

    ```bash
    product_sales = sales.groupby('ProductID')['Quantity'].sum()

    min_value = product_sales.min()

    product_sales[product_sales == min_value]
    ```

In [55]:
product_sales = sales.groupby('ProductID')['Quantity'].sum()

min_value = product_sales.min()

product_sales[product_sales == min_value]

ProductID
103    1
104    1
Name: Quantity, dtype: int64

In [33]:
sales[sales['ProductID']==102]

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price,TotalAmount
1,1002,2,102,2,Sarah,Kandy,Mouse,3000,6000
7,1008,3,102,3,Mike,Galle,Mouse,3000,9000


In [34]:
sales[sales['ProductID']==103]

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price,TotalAmount
2,1003,1,103,1,John,Colombo,Keyboard,8000,8000


In [36]:
sales[sales['ProductID']==104]

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price,TotalAmount
3,1004,3,104,1,Mike,Galle,Monitor,50000,50000


- Best selling product? Mouse
- Worst selling product? Keyboard and Monitor

# City Analysis

City
Colombo    508000
Galle       59000
Jaffna      24000
Kandy       18000
Name: TotalAmount, dtype: int64

# Top Orders

In [46]:
sales.groupby(['OrderID','CustomerName']).agg({'TotalAmount':'sum'}).sort_values(by='TotalAmount', ascending=False)

,,TotalAmount
OrderID,CustomerName,
1001,John,250000
1006,David,250000
1004,Mike,50000
1005,Emma,24000
1007,Sarah,12000
1008,Mike,9000
1003,John,8000
1002,Sarah,6000


# Multi Aggregation

In [47]:
sales.groupby("ProductName").agg({
    "Quantity":["sum","mean"],
    "TotalAmount":["sum","mean"]
})

Quantity      TotalAmount          
                 sum mean         sum      mean
ProductName                                    
Headphones         3  1.5       36000   18000.0
Keyboard           1  1.0        8000    8000.0
Laptop             2  1.0      500000  250000.0
Monitor            1  1.0       50000   50000.0
Mouse              5  2.5       15000    7500.0

In [48]:
sales

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price,TotalAmount
0,1001,1,101,1,John,Colombo,Laptop,250000,250000
1,1002,2,102,2,Sarah,Kandy,Mouse,3000,6000
2,1003,1,103,1,John,Colombo,Keyboard,8000,8000
3,1004,3,104,1,Mike,Galle,Monitor,50000,50000
4,1005,4,105,2,Emma,Jaffna,Headphones,12000,24000
5,1006,5,101,1,David,Colombo,Laptop,250000,250000
6,1007,2,105,1,Sarah,Kandy,Headphones,12000,12000
7,1008,3,102,3,Mike,Galle,Mouse,3000,9000


In [49]:
sales['Category'] = sales['TotalAmount'].apply(lambda x: 'Small Order' if x < 10000 else ('Medium Order' if x < 100000 else 'Large Order'))

In [50]:
sales

,OrderID,CustomerID,ProductID,Quantity,CustomerName,City,ProductName,Price,TotalAmount,Category
0,1001,1,101,1,John,Colombo,Laptop,250000,250000,Large Order
1,1002,2,102,2,Sarah,Kandy,Mouse,3000,6000,Small Order
2,1003,1,103,1,John,Colombo,Keyboard,8000,8000,Small Order
3,1004,3,104,1,Mike,Galle,Monitor,50000,50000,Medium Order
4,1005,4,105,2,Emma,Jaffna,Headphones,12000,24000,Medium Order
5,1006,5,101,1,David,Colombo,Laptop,250000,250000,Large Order
6,1007,2,105,1,Sarah,Kandy,Headphones,12000,12000,Medium Order
7,1008,3,102,3,Mike,Galle,Mouse,3000,9000,Small Order


In [53]:
sales.groupby('CustomerID').agg({'OrderID':'count','TotalAmount':['sum','mean']})

OrderID TotalAmount          
             count         sum      mean
CustomerID                              
1                2      258000  129000.0
2                2       18000    9000.0
3                2       59000   29500.0
4                1       24000   24000.0
5                1      250000  250000.0